# Hyperbolic Embedding Diagnostics

This notebook mirrors the functionality of `visualizations.ssl4eo.hyperbolic_visualization` using the original helper functions.

1. Update the configuration values below.
2. Set `RUN_PIPELINE = True`.
3. Execute the run cell to generate plots and summaries.

In [1]:
import math
import random
from types import SimpleNamespace
from pathlib import Path

import numpy as np
import torch
import torch.nn.functional as F
import sys
import matplotlib.pyplot as plt

from IPython.display import Image, display

# Ensure the open_clip_train helpers are importable when running inside the notebook.
repo_root = Path.cwd()
sys.path.insert(0, str(repo_root.parent.parent.parent))
from visualizations.ssl4eo import hyperbolic_visualization as hv


/home/juro4948/miniconda3/envs/ciip/lib/python3.12/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


In [2]:
# --- Required inputs ----------------------------------------------------
# --- Required arguments -------------------------------------------------
model_root = '/local/ms-data/SSL4EO/model/'
model_path = '2025_11_05-21_04_44-model_resnet50-lr_0.001-b_128-j_6-p_amp/'
checkpoint_root = Path(model_root) / model_path / "checkpoints"
output_dir = Path("diagnostics/output")

config_path = Path("path/to/config.yaml")
checkpoint_path = Path("path/to/checkpoint.pt")

# --- Optional overrides -------------------------------------------------
loss_checkpoint_path = None  # Path or None
output_dir = Path("hyperbolic_viz")
num_locations = 64
seed = 0
device_override = None  # e.g. "cuda:0" or "cpu"
curvature_override = None  # float or None
hyperbolic_eps = 1e-5
hyperbolic_normalize = True
cone_samples = 100
use_orthogonal_mapping = False
skip_final_fc = False
aperture_logk_override = None  # float or None

In [3]:
RUN_PIPELINE = True

if RUN_PIPELINE:
    device_str = device_override or ("cuda" if torch.cuda.is_available() else "cpu")
    device = torch.device(device_str)

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    config = hv.load_config(config_path)
    dataset = hv.prepare_dataset(config)
    indices, metadata = hv.sample_indices(dataset, num_locations, seed)

    checkpoint = hv.load_checkpoint(checkpoint_path, device)
    precision = str(getattr(config.model, "precision", "fp32"))
    input_dtype = hv.resolve_input_dtype(precision)

    model = hv.load_model_from_checkpoint(
        config,
        checkpoint_path,
        device=device,
        input_dtype=input_dtype,
        w_path=None,
        skip_final_fc=skip_final_fc,
        use_orthogonal_mapping=use_orthogonal_mapping,
    )

    loss_args = SimpleNamespace(
        hyperbolic=True,
        no_hyperbolic_normalize=not hyperbolic_normalize,
        curvature_init=1.0,
        hyperbolic_eps=hyperbolic_eps,
    )
    loss = hv.build_loss(loss_args, device)

    if loss_checkpoint_path is not None:
        extra_checkpoint = hv.load_checkpoint(loss_checkpoint_path, device)
        hv.maybe_load_loss_state(loss, extra_checkpoint)
    else:
        hv.maybe_load_loss_state(loss, checkpoint)

    hyp_params = hv.load_hyperbolic_params(checkpoint, hyperbolic_eps)
    loss.hyperbolic_eps = float(hyp_params["eps"])
    curvature_value = curvature_override if curvature_override is not None else hyp_params["c"]
    hv.override_curvature(loss, curvature_value, device)

    if hyp_params["hyp_scale_raw"] is not None:
        with torch.no_grad():
            loss.hyp_scale.copy_(loss.hyp_scale.new_tensor(hyp_params["hyp_scale_raw"]))

    s1_feats, s2_feats = hv.stack_features(model, dataset, indices, device)
    s1_feats_device = s1_feats.to(device)
    s2_feats_device = s2_feats.to(device)

    aperture_logk = (
        aperture_logk_override
        if aperture_logk_override is not None
        else (hyp_params["k_ap"] if hyp_params["k_ap"] is not None else math.log(0.5))
    )

    with torch.no_grad():
        context = hv.compute_hyperbolic_context(
            loss,
            s1_feats_device,
            s2_feats_device,
            aperture_logk=aperture_logk,
        )

    positive_angles = context["positive_angles"].cpu().numpy()
    aperture_s1 = context["aperture_s1"].cpu().numpy()
    aperture_s2 = context["aperture_s2"].cpu().numpy()
    s1_dirs = context["s1_dirs"].cpu().numpy()
    s2_dirs = context["s2_dirs"].cpu().numpy()
    s1_distances = context["s1_distances"].cpu().numpy()
    s2_distances = context["s2_distances"].cpu().numpy()

    with torch.no_grad():
        s1_prelift = loss._maybe_normalize(s1_feats_device)
        s2_prelift = loss._maybe_normalize(s2_feats_device)

    s1_norms = s1_prelift.norm(dim=-1).cpu().numpy()
    s2_norms = s2_prelift.norm(dim=-1).cpu().numpy()

    output_dir.mkdir(parents=True, exist_ok=True)

    hv.plot_angle_aperture(positive_angles, aperture_s1, aperture_s2, output_dir / "angle_aperture_scatter.png")
    hv.plot_radial_histogram(s1_norms, s2_norms, output_dir / "radial_histograms.png")
    hv.plot_angular_pca(s1_dirs, s2_dirs, s1_distances, s2_distances, output_dir / "angular_pca.png")
    hv.plot_cone_polar(
        positive_angles,
        aperture_s1,
        aperture_s2,
        output_dir / "cone_polar.png",
        cone_samples,
        seed,
    )

    records = []
    for idx, meta in enumerate(metadata):
        records.append(
            {
                **meta,
                "positive_angle": float(positive_angles[idx]),
                "aperture_s1": float(aperture_s1[idx]),
                "aperture_s2": float(aperture_s2[idx]),
                "s1_norm": float(s1_norms[idx]),
                "s2_norm": float(s2_norms[idx]),
                "s1_distance": float(s1_distances[idx]),
                "s2_distance": float(s2_distances[idx]),
            }
        )
    hv.write_metadata_csv(records, output_dir / "hyperbolic_summary.csv")

    used_curvature = loss._get_curvature(dtype=s1_feats.dtype, device=device).item()
    hyp_scale = float(F.softplus(loss.hyp_scale.detach()).item())
    aperture_scale = math.exp(aperture_logk) if aperture_logk is not None else float("nan")

    print(f"Saved plots to {output_dir.resolve()}")
    print(f"Effective curvature: {used_curvature:.6f}")
    print(
        "Hyperbolic params — eps: {eps:.2e}, hyp_scale: {scale:.6f}, aperture_scale: {ap:.6f} (logK={logk:.3f})".format(
            eps=loss.hyperbolic_eps,
            scale=hyp_scale,
            ap=aperture_scale,
            logk=aperture_logk if aperture_logk is not None else float('nan'),
        )
    )

    for image_name in [
        "angle_aperture_scatter.png",
        "radial_histograms.png",
        "angular_pca.png",
        "cone_polar.png",
    ]:
        image_path = output_dir / image_name
        if image_path.exists():
            display(Image(filename=str(image_path)))
else:
    print("Update the configuration cell above and set RUN_PIPELINE = True to execute.")

FileNotFoundError: [Errno 2] No such file or directory: '/home/juro4948/ciip/visualizations/ssl4eo/notebooks/path/to/config.yaml'